# Flood Risk Preliminary EDA
This notebook performs preliminary exploratory data analysis (EDA) on the flood-risk raster datasets and visualizes the Severn flood-risk target maps.

In [2]:

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (10,6)

#DATA_DIR = '/workspace/Data-Flood/'



terrain_severn = xr.open_dataset('flood_risk_terrain_severn.nc',engine='netcdf4')
terrain_northumbria = xr.open_dataset('flood_risk_terrain_northumbria.nc',engine='netcdf4')
# print(terrain_severn)


## Convert raster data to dataframe

In [3]:

df_s = terrain_severn.to_dataframe().reset_index()
df_n = terrain_northumbria.to_dataframe().reset_index()

print(df_s.shape)
df_s.head()


(83959808, 15)


,y,x,dtm,flow_dir,flow_acc,imd,waw,rciw,clc_type,risk_0_2m,risk_0_3m,risk_0_6m,risk_0_9m,risk_1_2m,spatial_ref
0,3186250.0,3413350.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,3186250.0,3413370.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,3186250.0,3413390.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,3186250.0,3413410.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,3186250.0,3413430.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


## Flood-risk target variables

In [ ]:

risk_cols = [
    'risk_0_2m',
    'risk_0_3m',
    'risk_0_6m',
    'risk_0_9m',
    'risk_1_2m'
]

df_s[risk_cols].describe()


## Missing values

In [ ]:

missing = df_s[risk_cols].isna().mean().sort_values(ascending=False) * 100
missing


## Elevation distribution

In [ ]:

df_s['dtm_m'] = df_s['dtm'] / 10

sns.histplot(df_s['dtm_m'].dropna(), bins=50)
plt.title('Severn Elevation Distribution')
plt.xlabel('Elevation (m)')
plt.show()


## Correlation between numerical variables

In [ ]:

num_cols = [
    'dtm',
    'flow_acc',
    'imd',
    'waw'
]

corr = df_s[num_cols].corr()

sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()


## Class distribution

In [ ]:

for col in risk_cols:
    plt.figure(figsize=(6,4))
    df_s[col].value_counts(dropna=False).sort_index().plot(kind='bar')
    plt.title(col)
    plt.xlabel('Class')
    plt.ylabel('Count')
    plt.show()


# Flood-risk maps for Severn

In [ ]:

def plot_risk_map(df, col, title):
    tmp = df[['x','y',col]].dropna()

    pivot = tmp.pivot_table(
        index='y',
        columns='x',
        values=col
    )

    plt.figure(figsize=(10,8))
    plt.imshow(
        pivot.values,
        origin='lower',
        cmap='Blues'
    )

    plt.colorbar(label='Flood Risk Class')
    plt.title(title)
    plt.xlabel('X')
    plt.ylabel('Y')
    plt.show()

for col in risk_cols:
    plot_risk_map(df_s, col, f'Severn Flood Risk Map — {col}')
